# Football-Synthesizer - Colab GPU runner

Offloads the two GPU-bound jobs from your RTX 3050 (4 GB) to Colab's **T4 (16 GB)**:
1. **Ball detector fine-tune** (`tools/finetune_ball.py`) - batch 16, fp16 -> minutes, not 18 h.
2. **Player tracking** (`tools/batch_match.py`) - detection + ByteTrack + calibration per chunk.

Everything else (possession linking, metrics, networks) is pure CPU - keep it local.

**Before you run:** `Runtime -> Change runtime type -> T4 GPU`. Put your `chunk_*.mp4` footage in a
Drive folder (copyright stays your call). Edit the **CONFIG** cell, then `Runtime -> Run all`.

## 0. CONFIG - edit these

In [ ]:
# --- where the code comes from -------------------------------------------------
# Option A: push the repo to GitHub and clone it (set the URL).
# Option B: leave GITHUB_URL = '' and point REPO_IN_DRIVE at a copy of the repo on Drive.
GITHUB_URL    = ''  # e.g. 'https://github.com/<you>/football-synthesizer.git'
REPO_IN_DRIVE = '/content/drive/MyDrive/football-synthesizer'  # used only if GITHUB_URL == ''

# --- your footage + outputs on Drive ------------------------------------------
DRIVE_FOOTAGE_DIR = '/content/drive/MyDrive/cv-football/chunks'   # holds chunk_*.mp4
DRIVE_OUT_DIR     = '/content/drive/MyDrive/cv-football/outputs'  # weights + parquets land here

# --- ball fine-tune inputs (annotations ship in the repo under outputs/ball_annotations) ------
FINETUNE_CHUNKS = ['chunk_000', 'chunk_001', 'chunk_006']  # those with click annotations
EPOCHS = 15
BATCH  = 16   # T4 has the VRAM for this (your 3050 was stuck at 2)

RUN_FINETUNE   = True
RUN_BATCHMATCH = True

## 1. GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

## 2. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Get the repo

In [ ]:
import os, subprocess, shutil
REPO = '/content/football-synthesizer'
if GITHUB_URL:
    if os.path.exists(REPO): shutil.rmtree(REPO)
    subprocess.run(['git', 'clone', '--depth', '1', GITHUB_URL, REPO], check=True)
else:
    assert os.path.exists(REPO_IN_DRIVE), f'no repo at {REPO_IN_DRIVE}; set GITHUB_URL or copy repo to Drive'
    if os.path.exists(REPO): shutil.rmtree(REPO)
    shutil.copytree(REPO_IN_DRIVE, REPO)
os.chdir(REPO)
print('repo at', REPO, '| contents:', os.listdir(REPO)[:20])

## 4. Install deps
Colab already has torch/numpy/pandas/opencv. We add the project's lighter deps + the CV stack.

In [ ]:
!pip -q install pyarrow 'mplsoccer>=1.2' tqdm pyyaml omegaconf ultralytics supervision \
    'torch-geometric>=2.5' scikit-learn jinja2 2>&1 | tail -3
print('deps installed')

## 5. WASB-SBDT (ball model factory + pretrained zoo)
`finetune_ball.py` reuses WASB's model factory and the `tracknetv2_soccer_best` checkpoint.

In [ ]:
WASB = '/content/WASB-SBDT'
if not os.path.exists(WASB):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/nttcom/WASB-SBDT.git', WASB], check=True)
os.environ['FOOTBALL_WASB_PATH'] = f'{WASB}/src'
# pretrained weights: place tracknetv2_soccer_best.pth.tar under WASB/pretrained_weights/.
# Keep a copy on Drive (DRIVE_OUT_DIR) and we copy it in:
PW = f'{WASB}/pretrained_weights'; os.makedirs(PW, exist_ok=True)
drive_w = f'{DRIVE_OUT_DIR}/tracknetv2_soccer_best.pth.tar'
if os.path.exists(drive_w):
    shutil.copy(drive_w, f'{PW}/tracknetv2_soccer_best.pth.tar')
print('WASB at', WASB, '| zoo present:', os.path.exists(f'{PW}/tracknetv2_soccer_best.pth.tar'))

## 6. Check footage

In [ ]:
assert os.path.isdir(DRIVE_FOOTAGE_DIR), f'no footage dir {DRIVE_FOOTAGE_DIR}'
mp4s = sorted(f for f in os.listdir(DRIVE_FOOTAGE_DIR) if f.endswith('.mp4'))
print(f'{len(mp4s)} chunks:', mp4s[:5], '...')
os.makedirs(f'{REPO}/outputs/ball_finetuned', exist_ok=True)

## 7. Ball fine-tune (T4, fp16, batch 16)

In [ ]:
if RUN_FINETUNE:
    env = dict(os.environ, PYTHONPATH='.', PYTHONUNBUFFERED='1')
    cmd = ['python', 'tools/finetune_ball.py', '--base', 'tracknetv2',
           '--epochs', str(EPOCHS), '--batch-size', str(BATCH),
           '--out', 'outputs/ball_finetuned/tracknetv2_colab.pth']
    for ck in FINETUNE_CHUNKS:
        cmd += ['--annotations', f'outputs/ball_annotations/{ck}.csv',
                '--video', f'{DRIVE_FOOTAGE_DIR}/{ck}.mp4']
    print(' '.join(cmd)); subprocess.run(cmd, env=env, check=True)

## 8. Player tracking across all chunks (`batch_match`)

In [ ]:
if RUN_BATCHMATCH:
    env = dict(os.environ, PYTHONPATH='.', PYTHONUNBUFFERED='1')
    cmd = ['python', 'tools/batch_match.py', '--chunks-dir', DRIVE_FOOTAGE_DIR,
           '--out-dir', 'outputs/match', '--sample-every', '5', '--skip-existing']
    print(' '.join(cmd)); subprocess.run(cmd, env=env, check=True)

## 9. Save outputs back to Drive
Pull the small artifacts (weights ~45 MB, parquets ~MBs) home; download or copy them locally.

In [ ]:
os.makedirs(DRIVE_OUT_DIR, exist_ok=True)
import glob
saved = []
for p in glob.glob(f'{REPO}/outputs/ball_finetuned/*.pth') + glob.glob(f'{REPO}/outputs/match/*.parquet'):
    dst = f'{DRIVE_OUT_DIR}/{os.path.basename(p)}'
    shutil.copy(p, dst); saved.append(dst)
print(f'saved {len(saved)} files to {DRIVE_OUT_DIR}:')
for s in saved: print(' ', s)

## Back on your laptop
1. Copy the weights + `outputs/match/*.parquet` from Drive into the repo.
2. Run the **CPU** half locally (no GPU pain):
```bash
PYTHONPATH=. python tools/ball_possession.py --video <chunk>.mp4 \
  --positions outputs/match/<chunk>.parquet --weights outputs/ball_finetuned/tracknetv2_colab.pth
PYTHONPATH=. python -m fingerprint.roles --positions outputs/match_anchored_dense.parquet --out outputs/roles_full.parquet
PYTHONPATH=. python tools/possession_report.py --positions <chunk>_dense.parquet \
  --ball <ball_track>.parquet --roles outputs/roles_full.parquet --chunk <chunk> --smooth
```